In [ ]:
import sys, torch, transformers, json
from pathlib import Path

print("python:", sys.version)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())

In [ ]:
from huggingface_hub import list_repo_files

BASE_TOKENIZER = "vinai/phobert-base-v2"
CKPT_REPO = "leduckhai/VietMed-NER"
CKPT_SUBFOLDER = "phobert-base-v2-VietMed-NER"

repo_files = [p for p in list_repo_files(CKPT_REPO) if p.startswith(f"{CKPT_SUBFOLDER}/")]
print("ckpt_repo:", CKPT_REPO)
print("ckpt_subfolder:", CKPT_SUBFOLDER)
print("files:", repo_files)

In [ ]:
from huggingface_hub import hf_hub_download

config_path = Path(
    hf_hub_download(
        repo_id=CKPT_REPO,
        filename="config.json",
        subfolder=CKPT_SUBFOLDER,
    )
)

print("downloaded_config:", config_path)

with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

print("model_type:", cfg.get("model_type"))
print("architectures:", cfg.get("architectures"))
print("num_labels:", cfg.get("num_labels"))
print("id2label:", cfg.get("id2label"))
print("label2id:", cfg.get("label2id"))

In [ ]:
id2label = cfg.get("id2label", {})
labels = [id2label[k] for k in sorted(id2label, key=lambda x: int(x) if str(x).isdigit() else str(x))]
print("labels:", labels)

generic = all(str(x).startswith("LABEL_") for x in labels)
print("generic_labels:", generic)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER, use_fast=True)
print(type(tokenizer))
print(tokenizer.__class__.__name__)

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    CKPT_REPO,
    subfolder=CKPT_SUBFOLDER,
)
print(type(model))
print(model.config.id2label)

In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

text = "Bệnh nhân dùng doxycycline điều trị viêm tuyến mồ hôi và làm xét nghiệm CRP."
preds = ner(text)
preds

In [ ]:
for p in preds:
    print(p)